# Sensor data preprocessing

Run the code cell below from the repository root. It validates the inputs, creates derived outputs in dependency order, runs focused tests and benchmarks, and records changes. Raw source CSVs are read-only inputs.

In [9]:
import json
import math
import os
import re
import shutil
import tempfile
import time
import unittest
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DERIVED = ROOT / "derived"
ENV_CSV = ROOT / "5EnvSensor_MayToDec2024_180kRows.csv"
OCC_CSV = ROOT / "5occupancySensor_MayToDec2024_9MRows.csv"
ENV_LOOKUP = ROOT / "Sensor ID and Locations - ENV.csv"
ENV_FULL = DERIVED / "environmental_measurements_wide.csv"
ENV_REDUCED = DERIVED / "environmental_measurements_wide_reduced.csv"
ENV_SCHEMA = DERIVED / "environmental_measurements_wide_schema.csv"
ENV_VALIDATION = DERIVED / "environmental_measurements_wide_validation.csv"
ENV_BENCHMARK = DERIVED / "environmental_measurements_wide_benchmark.csv"
OCC_STREAMS = DERIVED / "occupancy_streams_by_floorspace"
MISSING_REPORT = DERIVED / "preprocessing_missing_data_report.csv"
CHANGE_REPORT = DERIVED / "preprocessing_changes.csv"
VALIDATION_REPORT = DERIVED / "preprocessing_validation.csv"
BENCHMARK_REPORT = DERIVED / "preprocessing_benchmark.csv"
CHUNK = 50_000
SENSOR_ID = 6012000000000
DEVICE_ID = "56923747-b390-4f2a-8930-808429132266"
MISSING = {"", "nan", "null", "none"}
ENV_COLS = ["id", "sensorid", "devicetype", "jsondata", "status", "processing_errors", "createdate", "processdate"]
OCC_COLS = ["id", "deviceid", "floorspaceid", "occupancystatus", "headcount", "collecteddate", "occupancystatuschangedate", "previousoccupancystatus"]
validation, changes, missing_report, benchmarks = [], [], [], []


def add_validation(check, result, observed, expected, notes=""):
    validation.append({"check": check, "result": result, "observed": observed, "expected": expected, "notes": notes})


def add_change(dataset, source, output, operation, columns, reason, affected=0, before="", after=""):
    def relative(path):
        try:
            return Path(path).relative_to(ROOT).as_posix()
        except ValueError:
            return str(path)
    changes.append({"dataset": dataset, "source_file": relative(source), "output_file": relative(output), "operation": operation, "columns": columns, "reason": reason, "affected_rows": affected, "input_rows": before, "output_rows": after, "result": "PASS"})


def atomic_csv(frame, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", newline="", suffix=".csv", prefix=f".{destination.stem}.", dir=destination.parent, delete=False) as stream:
            temp = Path(stream.name)
            frame.to_csv(stream, index=False, lineterminator="\n", na_rep="NaN")
        os.replace(temp, destination)
        temp = None
    finally:
        if temp is not None and temp.exists():
            temp.unlink()


def numeric_or_nan(series, label):
    text = series.astype("string").str.strip()
    missing = text.isna() | text.str.casefold().isin(MISSING)
    numeric = pd.to_numeric(text.mask(missing), errors="coerce")
    bad = (~missing) & numeric.isna()
    if bad.any():
        raise ValueError(f"{label}: non-numeric value {text.loc[bad].iloc[0]!r}")
    if (numeric.dropna().map(math.isfinite) == False).any():
        raise ValueError(f"{label}: non-finite numeric value")
    return numeric, missing


def feature_slug(value):
    value = str(value).casefold().replace("2.5", "2_5")
    return re.sub(r"[^a-z0-9]+", "_", value).strip("_") or "unnamed"


def validate_env_lookup():
    lookup = pd.read_csv(ENV_LOOKUP, dtype=str, keep_default_na=False)
    if lookup.empty or not {"Sensor", "ID"}.issubset(lookup.columns):
        raise ValueError(f"{ENV_LOOKUP.name}: required Sensor/ID columns or rows missing")
    ids = pd.to_numeric(lookup["ID"], errors="coerce")
    if not lookup["Sensor"].eq("Env").all() or ids.isna().any() or not ids.eq(SENSOR_ID).all():
        raise ValueError(f"{ENV_LOOKUP.name}: unexpected sensor or ID")
    add_validation("ENV_location_lookup", "PASS", len(lookup), "Sensor=Env; ID=6012000000000")


def parse_json_payload(raw, record_id):
    try:
        payload = json.loads(raw)
    except (json.JSONDecodeError, TypeError) as exc:
        raise ValueError(f"Environmental record {record_id}: invalid JSON") from exc
    if not isinstance(payload, list):
        raise ValueError(f"Environmental record {record_id}: payload is not a list")
    measurements, metadata, times = {}, {}, set()
    for entry in payload:
        if not isinstance(entry, dict) or not isinstance(entry.get("variable"), dict):
            raise ValueError(f"Environmental record {record_id}: malformed measurement")
        variable = entry["variable"]
        var_id = str(variable.get("id", ""))
        if not var_id.isdigit() or var_id in measurements:
            raise ValueError(f"Environmental record {record_id}: invalid/duplicate variable ID")
        descriptor = {"variable_id": var_id, "variable": str(variable.get("name", "")), "unit": str(variable.get("unit", "")), "source": str(variable.get("source", "")), "structure": str(variable.get("structure", "")), "output_column": f"var_{var_id}_{feature_slug(variable.get('name', ''))}"}
        metadata[var_id] = descriptor
        points = entry.get("values")
        if not isinstance(points, list) or len(points) > 1:
            raise ValueError(f"Environmental record {record_id}: variable {var_id} must have at most one value point")
        if not points:
            measurements[var_id] = float("nan")
            continue
        if not isinstance(points[0], dict):
            raise ValueError(f"Environmental record {record_id}: variable {var_id} has a malformed value point")
        point = points[0]
        stamp = point.get("time")
        if stamp is None or isinstance(stamp, bool):
            raise ValueError(f"Environmental record {record_id}: missing measurement time")
        try:
            times.add(int(stamp))
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError(f"Environmental record {record_id}: invalid measurement time") from exc
        value = point.get("value")
        if value is None:
            measurements[var_id] = float("nan")
        else:
            try:
                value = float(value)
            except (TypeError, ValueError, OverflowError) as exc:
                raise ValueError(f"Environmental record {record_id}: non-numeric variable {var_id}") from exc
            if not math.isfinite(value):
                raise ValueError(f"Environmental record {record_id}: non-finite variable {var_id}")
            measurements[var_id] = value
    if len(times) != 1:
        raise ValueError(f"Environmental record {record_id}: expected one shared measurement timestamp")
    timestamp = datetime.fromtimestamp(next(iter(times)), tz=timezone.utc).isoformat().replace("+00:00", "Z")
    return timestamp, measurements, metadata


def build_environmental_outputs():
    headers = list(pd.read_csv(ENV_CSV, nrows=0, dtype=str, keep_default_na=False).columns)
    if set(headers) != set(ENV_COLS):
        raise ValueError(f"{ENV_CSV.name}: unexpected schema")
    descriptors, seen_ids = {}, set()
    row_count = equal_dates = timestamp_pairs = 0
    raw_missing = dict.fromkeys(ENV_COLS, 0)
    for chunk in pd.read_csv(ENV_CSV, dtype=str, keep_default_na=False, chunksize=CHUNK):
        row_count += len(chunk)
        for col in ENV_COLS:
            raw_missing[col] += int(chunk[col].str.strip().str.casefold().isin(MISSING).sum())
        ids = chunk["id"].astype(str)
        if ids.eq("").any() or ids.duplicated().any() or not seen_ids.isdisjoint(ids):
            raise ValueError("Environmental record IDs are blank or duplicated")
        seen_ids.update(ids)
        sensor = pd.to_numeric(chunk["sensorid"], errors="coerce")
        if sensor.isna().any() or not sensor.eq(SENSOR_ID).all():
            raise ValueError("Unexpected environmental sensor ID")
        for column, expected in (("devicetype", "env"), ("status", "Processed"), ("processing_errors", "NULL")):
            if not chunk[column].eq(expected).all():
                raise ValueError(f"Unexpected environmental {column}")
        created = pd.to_datetime(chunk["createdate"], errors="coerce", utc=True)
        processed = pd.to_datetime(chunk["processdate"], errors="coerce", utc=True)
        if created.isna().any() or processed.isna().any():
            raise ValueError("Invalid createdate/processdate")
        equal_dates += int(created.eq(processed).sum())
        timestamp_pairs += len(chunk)
        for record_id, raw in chunk[["id", "jsondata"]].itertuples(index=False, name=None):
            _, _, row_metadata = parse_json_payload(raw, record_id)
            for key, descriptor in row_metadata.items():
                if key in descriptors and descriptors[key] != descriptor:
                    raise ValueError(f"Environmental variable {key}: metadata changes between records")
                descriptors[key] = descriptor
    if row_count == 0 or not descriptors:
        raise ValueError("Environmental input is empty")
    variables = sorted(descriptors, key=int)
    features = [descriptors[key]["output_column"] for key in variables]
    if len(features) != len(set(features)):
        raise ValueError("Measurement variable names collide after normalization")
    schema = pd.DataFrame([descriptors[key] for key in variables])
    atomic_csv(schema, ENV_SCHEMA)

    columns = ["source_record_id", "sensor_id", "devicetype", "createdate", "processdate", "timestamp_utc", *features]
    missing_by_feature = dict.fromkeys(features, 0)
    temp = None
    written = 0
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", newline="", suffix=".csv", prefix=".environmental-wide.", dir=DERIVED, delete=False) as out:
            temp = Path(out.name)
            first = True
            for chunk in pd.read_csv(ENV_CSV, dtype=str, keep_default_na=False, chunksize=CHUNK):
                records = []
                for record_id, sensor, device, _, _, _, created, processed, raw_json in []:
                    pass
                for record in chunk.itertuples(index=False, name=None):
                    record_id, sensor, device, raw_json, _, _, created, processed = record
                    stamp, values, _ = parse_json_payload(raw_json, record_id)
                    item = {"source_record_id": record_id, "sensor_id": str(SENSOR_ID), "devicetype": device, "createdate": created, "processdate": processed, "timestamp_utc": stamp}
                    for key in variables:
                        item[descriptors[key]["output_column"]] = values.get(key, float("nan"))
                    records.append(item)
                expanded = pd.DataFrame(records, columns=columns)
                for feature in features:
                    normalized, is_missing = numeric_or_nan(expanded[feature], feature)
                    missing_by_feature[feature] += int(is_missing.sum())
                    expanded[feature] = normalized
                expanded.to_csv(out, index=False, header=first, lineterminator="\n", na_rep="NaN")
                first = False
                written += len(expanded)
        if written != row_count:
            raise ValueError(f"Wide output has {written} rows; expected {row_count}")
        os.replace(temp, ENV_FULL)
        temp = None
    finally:
        if temp is not None and temp.exists():
            temp.unlink()

    excluded = {"sensor_id", "devicetype"}
    if equal_dates == timestamp_pairs:
        excluded.add("processdate")
    selected = [col for col in columns if col not in excluded]
    temp = None
    reduced_rows = 0
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", newline="", suffix=".csv", prefix=".environmental-reduced.", dir=DERIVED, delete=False) as out:
            temp = Path(out.name)
            first = True
            for chunk in pd.read_csv(ENV_FULL, usecols=selected, dtype=str, keep_default_na=False, chunksize=CHUNK):
                for feature in features:
                    chunk[feature], _ = numeric_or_nan(chunk[feature], feature)
                chunk.to_csv(out, index=False, header=first, lineterminator="\n", na_rep="NaN")
                first = False
                reduced_rows += len(chunk)
        if reduced_rows != row_count:
            raise ValueError("Reduced environmental output changed the row count")
        os.replace(temp, ENV_REDUCED)
        temp = None
    finally:
        if temp is not None and temp.exists():
            temp.unlink()

    add_validation("environmental_rows_and_json", "PASS", f"{row_count:,} rows; {len(features)} variables", "one valid wide row per raw record")
    add_validation("environmental_ids", "PASS", len(seen_ids), row_count, "unique and nonblank")
    add_validation("environmental_constants", "PASS", "sensorid/devicetype/status/processing_errors", "all expected constants")
    add_validation("environmental_record_timestamps", "PASS" if equal_dates == timestamp_pairs else "INFO", f"{equal_dates:,}/{timestamp_pairs:,} equal", "parseable timestamps; equality required to drop processdate")
    add_change("environmental", ENV_CSV, ENV_FULL, "expand_json_to_wide", "jsondata -> measurement columns", "One wide row per source record; raw JSON left untouched.", row_count, row_count, written)
    add_change("environmental", ENV_CSV, ENV_FULL, "normalize_missing_measurements_to_NaN", ",".join(features), "Empty values arrays, JSON nulls, and absent variables become NaN in derived measurements.", sum(missing_by_feature.values()), row_count, written)
    add_change("environmental", ENV_FULL, ENV_REDUCED, "exclude_verified_columns", ",".join(sorted(excluded)), "Remove validated constants and equal processdate; retain measurement timestamp.", row_count, row_count, reduced_rows)
    for feature, count in missing_by_feature.items():
        missing_report.append({"dataset": "environmental", "column": feature, "missing_type": "missing_measurement", "count": count, "handling": "NaN in derived wide outputs; raw JSON and source rows unchanged."})
    return {"rows": row_count, "features": features, "columns": columns, "selected": selected, "excluded": sorted(excluded), "missing": missing_by_feature, "timestamp_equal": equal_dates}


def validate_wide_file(path, raw_rows, reduced):
    schema = pd.read_csv(ENV_SCHEMA, dtype=str, keep_default_na=False)
    features = schema["output_column"].tolist()
    headers = list(pd.read_csv(path, nrows=0, dtype=str, keep_default_na=False).columns)
    expected = set(["source_record_id", "createdate", "timestamp_utc", *features])
    if not expected.issubset(headers):
        raise ValueError(f"{path.name}: required output columns missing")
    if not reduced and not {"sensor_id", "devicetype", "processdate"}.issubset(headers):
        raise ValueError(f"{path.name}: full output columns missing")
    ids, rows = set(), 0
    for chunk in pd.read_csv(path, dtype=str, keep_default_na=False, chunksize=CHUNK):
        rows += len(chunk)
        chunk_ids = chunk["source_record_id"].astype(str)
        if chunk_ids.eq("").any() or chunk_ids.duplicated().any() or not ids.isdisjoint(chunk_ids):
            raise ValueError(f"{path.name}: duplicate/blank source record ID")
        ids.update(chunk_ids)
        if pd.to_datetime(chunk["timestamp_utc"], errors="coerce", utc=True).isna().any():
            raise ValueError(f"{path.name}: invalid measurement timestamp")
        for feature in features:
            numeric_or_nan(chunk[feature], feature)
    if rows != raw_rows or len(ids) != raw_rows:
        raise ValueError(f"{path.name}: row/ID count differs from raw input")
    add_validation(f"{path.stem}_schema_rows_ids", "PASS", f"{rows:,} rows; {len(headers)} columns", f"{raw_rows:,} matching unique IDs", "Measurement fields are numeric or NaN.")


def audit_change_date(path):
    data = pd.read_csv(path, usecols=["id", "headcount", "collecteddate", "occupancystatuschangedate"], dtype=str, keep_default_na=False)
    ids = pd.to_numeric(data["id"], errors="coerce")
    count, _ = numeric_or_nan(data["headcount"], "headcount")
    collected = pd.to_datetime(data["collecteddate"], errors="coerce", utc=True)
    changed = pd.to_datetime(data["occupancystatuschangedate"], errors="coerce", utc=True)
    if ids.isna().any() or collected.isna().any() or changed.isna().any():
        raise ValueError(f"{path.name}: invalid ID or timestamp in change-date audit")
    data = data.assign(_id=ids, _count=count, _collected=collected, _changed=changed).sort_values(["_collected", "_id"], kind="stable").reset_index(drop=True)
    previous_count, previous_changed = data["_count"].shift(), data["_changed"].shift()
    zero, positive = data["_count"].eq(0), data["_count"].gt(0)
    starts, runs = positive & previous_count.eq(0), positive & previous_count.gt(0)
    zero_match = zero & data["_changed"].eq(data["_collected"])
    start_match = starts & data["_changed"].eq(data["_collected"])
    run_match = runs & data["_changed"].eq(previous_changed)
    start_bad = starts & ~data["_changed"].eq(data["_collected"])
    run_bad = runs & ~data["_changed"].eq(previous_changed)
    return {"floorspaceid": path.stem, "rows": len(data), "zero_readings": int(zero.sum()), "zero_time_matches": int(zero_match.sum()), "positive_starts_after_zero": int(starts.sum()), "positive_start_time_matches": int(start_match.sum()), "positive_start_first_exception_id": str(data.loc[start_bad, "id"].iloc[0]) if start_bad.any() else "", "successive_positive_pairs": int(runs.sum()), "unchanged_time_in_positive_run": int(run_match.sum()), "positive_run_time_changes": int(run_bad.sum()), "positive_run_first_exception_id": str(data.loc[run_bad, "id"].iloc[0]) if run_bad.any() else ""}


def build_occupancy_streams():
    headers = list(pd.read_csv(OCC_CSV, nrows=0, dtype=str, keep_default_na=False).columns)
    if set(headers) != set(OCC_COLS):
        raise ValueError(f"{OCC_CSV.name}: unexpected source schema")
    temp_root = Path(tempfile.mkdtemp(prefix=".occupancy-preprocessing."))
    staging, final = temp_root / "staging", temp_root / "final"
    staging.mkdir()
    final.mkdir()
    rows_by_space, source_rows = {}, 0
    unknown = blank_previous = timestamp_equal = 0
    raw_missing = dict.fromkeys(OCC_COLS, 0)
    count_min = count_max = None
    try:
        for chunk in pd.read_csv(OCC_CSV, dtype=str, keep_default_na=False, chunksize=CHUNK):
            source_rows += len(chunk)
            for col in OCC_COLS:
                raw_missing[col] += int(chunk[col].str.strip().str.casefold().isin(MISSING).sum())
            if not chunk["deviceid"].eq(DEVICE_ID).all():
                raise ValueError("Unexpected occupancy deviceid")
            if chunk["floorspaceid"].str.strip().str.casefold().isin(MISSING).any():
                raise ValueError("Occupancy floorspaceid is missing")
            headcount, _ = numeric_or_nan(chunk["headcount"], "occupancy headcount")
            valid = headcount.dropna()
            if (valid < 0).any() or ((valid % 1) != 0).any():
                raise ValueError("Occupancy headcount must be a nonnegative integer or missing")
            if not valid.empty:
                count_min = valid.min() if count_min is None else min(count_min, valid.min())
                count_max = valid.max() if count_max is None else max(count_max, valid.max())
            collected = pd.to_datetime(chunk["collecteddate"], errors="coerce", utc=True)
            changed = pd.to_datetime(chunk["occupancystatuschangedate"], errors="coerce", utc=True)
            if collected.isna().any() or changed.isna().any():
                raise ValueError("Occupancy timestamps contain invalid values")
            timestamp_equal += int(collected.eq(changed).sum())
            previous = chunk["previousoccupancystatus"].str.strip().str.casefold()
            unknown += int(previous.eq("unknown").sum())
            blank_previous += int(previous.isin(MISSING).sum())
            chunk["headcount"] = headcount
            for space, group in chunk[["id", "floorspaceid", "headcount", "collecteddate", "occupancystatuschangedate"]].groupby("floorspaceid", sort=False):
                if any(char in space for char in '<>:"/\\|?*') or space.rstrip(" .") != space:
                    raise ValueError(f"Invalid floorspace ID for output filename {space!r}")
                path = staging / f"{space}.csv"
                group.to_csv(path, mode="a" if path.exists() else "w", header=not path.exists(), index=False, lineterminator="\n", na_rep="NaN")
                rows_by_space[space] = rows_by_space.get(space, 0) + len(group)
        if source_rows == 0:
            raise ValueError("Occupancy source is empty")
        audits = []
        for space in sorted(rows_by_space):
            source = staging / f"{space}.csv"
            audit = audit_change_date(source)
            if audit["rows"] != rows_by_space[space]:
                raise ValueError(f"{source.name}: change-date audit row mismatch")
            audits.append(audit)
            target = final / source.name
            first, count = True, 0
            for chunk in pd.read_csv(source, dtype=str, keep_default_na=False, chunksize=CHUNK):
                chunk.drop(columns=["occupancystatuschangedate"]).to_csv(target, mode="w" if first else "a", header=first, index=False, lineterminator="\n", na_rep="NaN")
                count += len(chunk)
                first = False
            if count != rows_by_space[space]:
                raise ValueError(f"{target.name}: output row count mismatch")
        pd.DataFrame([{"floorspaceid": key, "row_count": value} for key, value in rows_by_space.items()]).to_csv(final / "stream_row_counts.csv", index=False)
        audit_frame = pd.DataFrame(audits)
        audit_frame.to_csv(final / "occupancystatuschangedate_audit.csv", index=False)
        if OCC_STREAMS.exists():
            staged_names = {path.name for path in final.iterdir()}
            if any(path.is_dir() for path in OCC_STREAMS.iterdir()):
                raise ValueError(f"Unexpected subdirectory in derived occupancy output: {OCC_STREAMS}")
            for staged_file in final.iterdir():
                os.replace(staged_file, OCC_STREAMS / staged_file.name)
            for stale_file in OCC_STREAMS.iterdir():
                if stale_file.name not in staged_names:
                    stale_file.unlink()
        else:
            final.rename(OCC_STREAMS)
        add_validation("occupancy_rows_and_spaces", "PASS", f"{source_rows:,} rows; {len(rows_by_space)} spaces", "all source rows retained")
        add_validation("occupancy_device_constant", "PASS", DEVICE_ID, "constant across all rows")
        add_validation("occupancy_timestamps", "PASS", f"all parseable; {timestamp_equal:,}/{source_rows:,} equal", "parseable; equality reported")
        add_validation("occupancy_headcount", "PASS", f"numeric integer range {count_min}-{count_max}", "nonnegative integer or NaN")
        add_validation("occupancy_previous_unknown", "INFO", unknown, "counted and omitted from derived streams")
        add_change("occupancy", OCC_CSV, OCC_STREAMS, "split_by_floorspace", "deviceid,occupancystatus,previousoccupancystatus", "Validate deviceid; omit validated constant ID and category/history fields from selected derived series.", source_rows, source_rows, source_rows)
        add_change("occupancy", OCC_CSV, OCC_STREAMS, "count_and_exclude_unknown_history", "previousoccupancystatus", "Unknown values are counted in the missingness report; the history column is omitted from derived streams.", unknown, source_rows, source_rows)
        add_change("occupancy", OCC_CSV, OCC_STREAMS, "normalize_missing_headcount_to_NaN", "headcount", "Missing headcounts remain as NaN without dropping stream rows.", raw_missing["headcount"], source_rows, source_rows)
        add_change("occupancy", OCC_CSV, OCC_STREAMS, "audit_then_drop_change_timestamp", "occupancystatuschangedate", "Write pattern audit before removing the timestamp from derived series.", source_rows, source_rows, source_rows)
        missing_report.extend([
            {"dataset": "occupancy", "column": "previousoccupancystatus", "missing_type": "unknown", "count": unknown, "handling": "Omitted from derived streams; raw CSV unchanged."},
            {"dataset": "occupancy", "column": "previousoccupancystatus", "missing_type": "blank_or_null", "count": blank_previous, "handling": "Omitted from derived streams; raw CSV unchanged."},
            {"dataset": "occupancy", "column": "headcount", "missing_type": "blank_or_null", "count": raw_missing["headcount"], "handling": "NaN in derived stream; rows retained."},
        ])
        return {"rows": source_rows, "spaces": len(rows_by_space), "unknown": unknown, "audit": audit_frame, "selected": ["id", "floorspaceid", "headcount", "collecteddate"]}
    finally:
        if temp_root.exists():
            shutil.rmtree(temp_root)


def benchmark_file(path, dataset, reduced_columns=None):
    headers = list(pd.read_csv(path, nrows=0, dtype=str, keep_default_na=False).columns)
    results = []
    for kind, columns in (("full_read", None), ("reduced_read", reduced_columns or headers)):
        start, count, largest = time.perf_counter(), 0, 0
        for chunk in pd.read_csv(path, usecols=columns, chunksize=CHUNK):
            count += len(chunk)
            largest = max(largest, int(chunk.memory_usage(index=True, deep=True).sum()))
        results.append({"dataset": dataset, "read_type": kind, "rows": count, "columns": len(headers) if columns is None else len(columns), "elapsed_seconds": time.perf_counter() - start, "largest_chunk_mib": largest / 2**20})
    if results[0]["rows"] != results[1]["rows"]:
        raise ValueError(f"{path.name}: full and reduced benchmark row counts differ")
    benchmarks.extend(results)


class PreprocessingTests(unittest.TestCase):
    def test_missing_values_are_nan_not_zero(self):
        numbers, missing = numeric_or_nan(pd.Series(["", "NULL", "nan", "2.5"]), "test")
        self.assertEqual(int(missing.sum()), 3)
        self.assertTrue(pd.isna(numbers.iloc[0]))
        self.assertEqual(numbers.iloc[3], 2.5)

    def test_bad_numeric_is_rejected(self):
        with self.assertRaises(ValueError):
            numeric_or_nan(pd.Series(["unknown"]), "test")

    def test_slug_is_predictable(self):
        self.assertEqual(feature_slug("Particulate matter 2.5"), "particulate_matter_2_5")


DERIVED.mkdir(parents=True, exist_ok=True)
for required_file in (ENV_CSV, OCC_CSV, ENV_LOOKUP):
    if not required_file.is_file():
        raise FileNotFoundError(f"Required input is missing: {required_file}")
tests = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(PreprocessingTests))
if not tests.wasSuccessful():
    raise AssertionError("Preprocessing tests failed")
validate_env_lookup()
env = build_environmental_outputs()
validate_wide_file(ENV_FULL, env["rows"], reduced=False)
validate_wide_file(ENV_REDUCED, env["rows"], reduced=True)
add_validation("environmental_missing_cells", "INFO", sum(env["missing"].values()), "preserved as NaN", f"{sum(value > 0 for value in env['missing'].values())} variables")
environmental_validation = validation.copy()
occ = build_occupancy_streams()
benchmark_file(ENV_FULL, ENV_FULL.name, env["selected"])
benchmark_file(ENV_REDUCED, ENV_REDUCED.name)
benchmark_file(OCC_CSV, OCC_CSV.name, occ["selected"])

wide_bench = pd.DataFrame([row for row in benchmarks if row["dataset"] in {ENV_FULL.name, ENV_REDUCED.name}])
atomic_csv(wide_bench, ENV_BENCHMARK)
atomic_csv(pd.DataFrame(environmental_validation), ENV_VALIDATION)
atomic_csv(pd.DataFrame(validation), VALIDATION_REPORT)
atomic_csv(pd.DataFrame(missing_report), MISSING_REPORT)
atomic_csv(pd.DataFrame(changes), CHANGE_REPORT)
atomic_csv(pd.DataFrame(benchmarks), BENCHMARK_REPORT)
print(f"Environmental: {env['rows']:,} rows -> {len(env['features'])} variables; {sum(env['missing'].values()):,} missing cells encoded as NaN.")
print(f"Occupancy: {occ['rows']:,} rows retained across {occ['spaces']} floorspaces; {occ['unknown']:,} previous-status Unknown values counted and omitted.")
print(f"Outputs and audit reports written to {DERIVED}.")


test_bad_numeric_is_rejected (__main__.PreprocessingTests.test_bad_numeric_is_rejected) ... ok
test_missing_values_are_nan_not_zero (__main__.PreprocessingTests.test_missing_values_are_nan_not_zero) ... ok
test_slug_is_predictable (__main__.PreprocessingTests.test_slug_is_predictable) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.007s

OK


Environmental: 179,447 rows -> 11 variables; 149,782 missing cells encoded as NaN.
Occupancy: 9,091,732 rows retained across 5 floorspaces; 193,058 previous-status Unknown values counted and omitted.
Outputs and audit reports written to c:\Users\joube\OneDrive - Monash University\Year 5 - Semester 2\MMA3001\MMA3001-Project\derived.
